<a href="https://colab.research.google.com/github/sahilmohammed-ai/building-with-anthropic-api/blob/main/prompt_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Prompt Evaluation

In [ ]:
%pip install anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.1/472.1 kB 5.0 MB/s eta 0:00:00


In [1]:
from google.colab import userdata
api_key = userdata.get('ANTHROPIC_API_KEY').strip()

In [2]:
from anthropic import Anthropic
client = Anthropic(api_key=api_key)
model = "claude-haiku-4-5"

In [3]:
# helper functions
def add_user_message(messages, text):
  user_message = {'role': 'user', 'content': text}
  messages.append(user_message)

def add_assistant_message(messages, text):
  assistant_message = {'role': 'assistant', 'content': text}
  messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=None):
  params = {
    'model':model,
    'max_tokens':1000,
    'messages':messages,
    'temperature':temperature
  }
  if system:
    params['system'] = system

  if stop_sequences:
    params['stop_sequences'] = stop_sequences

  message = client.messages.create(**params)
  return message.content[0].text

In [4]:
# eval dataset generation
import json

def generate_dataset():
  prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete. Also, ask for a certain solution
criteria to build an effective response.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex",
        "solution_criteria": Must include (solution criteria)
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 5 objects.
"""

  messages = []
  add_user_message(messages, prompt)
  add_assistant_message(messages, '```json')
  text = chat(messages, stop_sequences=['```'])
  return json.loads(text)

In [5]:
# save dataset file
dataset = generate_dataset()

with open('dataset.json', 'w') as f:
  json.dump(dataset, f, indent=2)

In [10]:
# model-based grading
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Criteria to Evaluate Solution on:
<criteria>
{test_case["solution_criteria"]}
</criteria>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [7]:
# code-based grading
import re # regex syntax helper module
import ast # python syntax helper module

def validate_json(text):
  try:
    json.loads(text.strip())
    return 10
  except json.JSONDecodeError:
    return 0

def validate_python(text):
  try:
    ast.parse(text.strip())
    return 10
  except SyntaxError:
    return 0

def validate_regex(text):
  try:
    re.compile(text.strip())
    return 10
  except re.error:
    return 0

def grade_syntax(response, test_case):
  format = test_case['format']
  if format == 'json':
    return validate_json(response)
  elif format == 'python':
    return validate_python(response)
  else:
    return validate_regex(response)

In [8]:
# more helper functions
from statistics import mean

def run_prompt(test_case):
  prompt =f"""
    Please solve the following task:
    {test_case['task']}

    * Respond with only Python, JSON, or a plain Regex
    * Do not add any comments, commentary, or explainations
    """

  messages = []
  add_user_message(messages, prompt)
  add_assistant_message(messages, '```code')
  output = chat(messages, stop_sequences=['```'])
  return output

def run_test_case(test_case):
  output = run_prompt(test_case)
  model_grade = grade_by_model(test_case, output)
  model_score = model_grade['score']
  reasoning = model_grade['reasoning']
  syntax_score = grade_syntax(output, test_case)
  score = mean([model_score, syntax_score])

  return {
      'output': output,
      'test_case': test_case,
      'score': score,
      'reasoning': reasoning,
  }

def run_eval(dataset):
  results = []

  for test_case in dataset:
    result = run_test_case(test_case)
    results.append(result)

  average_score = mean([result['score'] for result in results])
  print(f'Average Score: {average_score}')

  return results

In [11]:
with open('dataset.json', 'r') as f:
  dataset = json.load(f)
results = run_eval(dataset)

Average Score: 7.0


In [12]:
print(json.dumps(results, indent=2))

[
  {
    "output": "\nimport json\nimport sys\n\ndef parse_cloudformation_template(template_content):\n    try:\n        template = json.loads(template_content)\n    except json.JSONDecodeError:\n        return {}\n    \n    resources = template.get('Resources', {})\n    result = {}\n    \n    for logical_id, resource_details in resources.items():\n        resource_type = resource_details.get('Type', 'Unknown')\n        result[logical_id] = resource_type\n    \n    return result\n\nif __name__ == '__main__':\n    template_input = sys.stdin.read()\n    parsed_resources = parse_cloudformation_template(template_input)\n    print(json.dumps(parsed_resources, indent=2))\n",
    "test_case": {
      "task": "Parse an AWS CloudFormation template to extract all resource logical IDs and their types",
      "format": "python",
      "solution_criteria": "Must read JSON input, iterate through Resources section, return a dictionary mapping logical IDs to resource types"
    },
    "score": 8.5,
 